# kafka_spark.example.ipynb
## Full Pipeline Demo: Real-Time Stock Market Pipeline
**Author**: Aashish Vinod  
**Course**: DATA605 Spring 2026

## Pipeline Steps
1. Setup
2. Simulate and explore stock data
3. Publish events to Kafka
4. Consume events from Kafka
5. Compute moving averages
6. Generate price alerts
7. Visualize results
8. Performance analysis

## Step 1: Setup

In [ ]:
import json
import time
import random
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from kafka import KafkaProducer, KafkaConsumer
from kafka.admin import KafkaAdminClient, NewTopic
from kafka.errors import TopicAlreadyExistsError
from kafka_spark_utils import (
    generate_stock_event, compute_moving_average,
    check_price_alert, format_kafka_summary,
    serialize_event, deserialize_event,
    STOCK_SYMBOLS, BASE_PRICES,
)

KAFKA_BROKER = 'localhost:9092'
TOPIC_NAME = 'stock-prices'
N_EVENTS = 200
sns.set_style('darkgrid')
plt.rcParams['figure.figsize'] = (12, 5)
print('Setup complete!')
print(f'Kafka Broker : {KAFKA_BROKER}')
print(f'Topic        : {TOPIC_NAME}')
print(f'Stocks       : {STOCK_SYMBOLS}')


## Step 2: Explore Simulated Stock Data

In [ ]:
# Generate sample events for exploration (no Kafka needed)
sample_events = []
for symbol in STOCK_SYMBOLS:
    for _ in range(40):
        sample_events.append(generate_stock_event(symbol))

df_sample = pd.DataFrame(sample_events)
df_sample['timestamp'] = pd.to_datetime(df_sample['timestamp'])
print(f'Generated {len(df_sample)} sample events')
print(df_sample.head(10))


In [ ]:
# Summary statistics
summary = df_sample.groupby('symbol').agg(
    count=('price', 'count'),
    avg_price=('price', 'mean'),
    min_price=('price', 'min'),
    max_price=('price', 'max'),
    avg_volume=('volume', 'mean'),
).round(2)
print('Summary Statistics by Stock Symbol:')
print(summary)


In [ ]:
# Visualize simulated price distributions
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for symbol in STOCK_SYMBOLS:
    prices = df_sample[df_sample['symbol'] == symbol]['price']
    axes[0].hist(prices, alpha=0.6, label=symbol, bins=12)
axes[0].set_title('Simulated Stock Price Distributions')
axes[0].set_xlabel('Price ($)')
axes[0].set_ylabel('Frequency')
axes[0].legend()
df_sample.groupby('symbol')['volume'].mean().plot(kind='bar', ax=axes[1], color='steelblue')
axes[1].set_title('Average Volume by Stock Symbol')
axes[1].set_xlabel('Symbol')
axes[1].set_ylabel('Average Volume')
axes[1].tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.savefig('stock_distributions.png', dpi=100, bbox_inches='tight')
plt.show()
print('Plot saved as stock_distributions.png')


## Step 3: Publish Events to Kafka

In [ ]:
# Create Kafka topic
admin_client = KafkaAdminClient(bootstrap_servers=KAFKA_BROKER)
try:
    topic = NewTopic(name=TOPIC_NAME, num_partitions=3, replication_factor=1)
    admin_client.create_topics([topic])
    print(f'Topic "{TOPIC_NAME}" created!')
except TopicAlreadyExistsError:
    print(f'Topic "{TOPIC_NAME}" already exists.')
finally:
    admin_client.close()


In [ ]:
# Publish stock events to Kafka
producer = KafkaProducer(
    bootstrap_servers=KAFKA_BROKER,
    value_serializer=lambda v: json.dumps(v).encode('utf-8'),
    key_serializer=lambda k: k.encode('utf-8'),
)
produced_events = []
start_time = time.time()
print(f'Publishing {N_EVENTS} events to Kafka...')
for i in range(N_EVENTS):
    symbol = random.choice(STOCK_SYMBOLS)
    event = generate_stock_event(symbol)
    producer.send(TOPIC_NAME, key=symbol, value=event)
    produced_events.append(event)
    if (i + 1) % 50 == 0:
        print(f'  Published {i + 1}/{N_EVENTS} events...')
producer.flush()
producer.close()
elapsed = time.time() - start_time
print(f'Done! {N_EVENTS} events in {elapsed:.2f}s ({N_EVENTS/elapsed:.1f} events/sec)')


## Step 4: Consume Events from Kafka

In [ ]:
# Consume all events
consumer = KafkaConsumer(
    TOPIC_NAME,
    bootstrap_servers=KAFKA_BROKER,
    auto_offset_reset='earliest',
    enable_auto_commit=True,
    group_id='stock-analysis-group',
    value_deserializer=lambda x: json.loads(x.decode('utf-8')),
    consumer_timeout_ms=5000,
)
consumed_events = []
start_time = time.time()
for msg in consumer:
    consumed_events.append(msg.value)
consumer.close()
elapsed = time.time() - start_time
print(f'Consumed {len(consumed_events)} events in {elapsed:.2f}s')
print()
print(format_kafka_summary(consumed_events))


## Step 5: Moving Averages (Spark-style Aggregation)

In [ ]:
# Build DataFrame from consumed events
df = pd.DataFrame(consumed_events)
df['timestamp'] = pd.to_datetime(df['timestamp'])
df = df.sort_values('timestamp').reset_index(drop=True)
print(f'DataFrame shape: {df.shape}')
print(df.head())


In [ ]:
# Compute moving averages per symbol
results = {}
for symbol in STOCK_SYMBOLS:
    df_sym = df[df['symbol'] == symbol].copy().reset_index(drop=True)
    prices = df_sym['price'].tolist()
    df_sym['MA5']  = compute_moving_average(prices, window=5)
    df_sym['MA10'] = compute_moving_average(prices, window=10)
    results[symbol] = df_sym
    last_ma5 = df_sym['MA5'].dropna().iloc[-1] if df_sym['MA5'].dropna().any() else 'N/A'
    print(f'{symbol}: {len(prices)} events | last price: ${prices[-1]:.2f} | MA5: ${last_ma5}')


In [ ]:
# Plot moving averages
fig, axes = plt.subplots(len(STOCK_SYMBOLS), 1, figsize=(14, 4 * len(STOCK_SYMBOLS)))
for idx, symbol in enumerate(STOCK_SYMBOLS):
    df_sym = results[symbol]
    ax = axes[idx]
    ax.plot(df_sym.index, df_sym['price'], label='Price', alpha=0.5, linewidth=1)
    ax.plot(df_sym.index, df_sym['MA5'],   label='MA5',   linewidth=2, color='orange')
    ax.plot(df_sym.index, df_sym['MA10'],  label='MA10',  linewidth=2, color='red')
    ax.axhline(y=BASE_PRICES[symbol], color='gray', linestyle='--', alpha=0.5, label='Base')
    ax.set_title(f'{symbol} - Price with Moving Averages')
    ax.set_xlabel('Event Index')
    ax.set_ylabel('Price ($)')
    ax.legend()
plt.tight_layout()
plt.savefig('moving_averages.png', dpi=100, bbox_inches='tight')
plt.show()
print('Plot saved as moving_averages.png')


## Step 6: Price Alert Generation

In [ ]:
# Generate price alerts
all_alerts = []
for event in consumed_events:
    alert = check_price_alert(event['price'], event['symbol'], threshold_pct=1.0)
    if alert:
        all_alerts.append(alert)

print(f'Total alerts : {len(all_alerts)} out of {len(consumed_events)} events')
print(f'Alert rate   : {len(all_alerts)/len(consumed_events)*100:.1f}%')
if all_alerts:
    df_alerts = pd.DataFrame(all_alerts)
    print('\nAlerts by symbol:')
    print(df_alerts.groupby('symbol').size().reset_index(name='alert_count'))
    print('\nSample alerts:')
    print(df_alerts.head(5).to_string())


In [ ]:
# Visualize alerts
if all_alerts:
    df_alerts = pd.DataFrame(all_alerts)
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    df_alerts.groupby('symbol').size().plot(kind='bar', ax=axes[0], color='tomato')
    axes[0].set_title('Price Alerts by Stock Symbol')
    axes[0].set_xlabel('Symbol')
    axes[0].set_ylabel('Number of Alerts')
    axes[0].tick_params(axis='x', rotation=0)
    axes[1].hist(df_alerts['change_pct'], bins=20, color='steelblue', edgecolor='black')
    axes[1].set_title('Distribution of Price Change % in Alerts')
    axes[1].set_xlabel('Change %')
    axes[1].set_ylabel('Count')
    plt.tight_layout()
    plt.savefig('price_alerts.png', dpi=100, bbox_inches='tight')
    plt.show()
    print('Plot saved as price_alerts.png')


## Step 7: Performance Analysis

In [ ]:
# Throughput test: produce 500 events and measure time
producer = KafkaProducer(
    bootstrap_servers=KAFKA_BROKER,
    value_serializer=lambda v: json.dumps(v).encode('utf-8'),
    key_serializer=lambda k: k.encode('utf-8'),
)
batch_sizes = [50, 100, 200, 500]
throughputs = []
for batch in batch_sizes:
    start = time.time()
    for _ in range(batch):
        symbol = random.choice(STOCK_SYMBOLS)
        event = generate_stock_event(symbol)
        producer.send(TOPIC_NAME, key=symbol, value=event)
    producer.flush()
    elapsed = time.time() - start
    tput = batch / elapsed
    throughputs.append(tput)
    print(f'Batch {batch:4d}: {elapsed:.3f}s | {tput:.1f} events/sec')
producer.close()


In [ ]:
# Visualize throughput
plt.figure(figsize=(8, 4))
plt.plot(batch_sizes, throughputs, marker='o', color='steelblue', linewidth=2)
plt.title('Kafka Producer Throughput vs Batch Size')
plt.xlabel('Batch Size (number of events)')
plt.ylabel('Throughput (events/sec)')
plt.grid(True)
plt.tight_layout()
plt.savefig('throughput.png', dpi=100, bbox_inches='tight')
plt.show()
print('Throughput plot saved!')


## Step 8: Summary and Results

In [ ]:
print('=' * 60)
print('PIPELINE SUMMARY')
print('=' * 60)
print(f'Total events produced : {N_EVENTS}')
print(f'Total events consumed : {len(consumed_events)}')
print(f'Total alerts triggered: {len(all_alerts)}')
print(f'Alert rate            : {len(all_alerts)/len(consumed_events)*100:.1f}%')
print()
print('Moving Average Results (last values):')
for symbol in STOCK_SYMBOLS:
    df_sym = results[symbol]
    last_price = df_sym['price'].iloc[-1]
    last_ma5   = df_sym['MA5'].dropna().iloc[-1] if len(df_sym['MA5'].dropna()) > 0 else float('nan')
    last_ma10  = df_sym['MA10'].dropna().iloc[-1] if len(df_sym['MA10'].dropna()) > 0 else float('nan')
    print(f'  {symbol}: price=${last_price:.2f} | MA5=${last_ma5:.2f} | MA10=${last_ma10:.2f}')
print()
print('Conclusion:')
print('  - Apache Kafka successfully ingested real-time stock price events')
print('  - Moving averages smoothed out short-term price fluctuations')
print('  - Price alerts identified stocks with significant movements')
print('  - The pipeline is scalable and runs fully locally via Docker')
